# LeetCode 1888 — Minimum Number of Flips to Make the Binary String Alternating bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-flips-to-make-the-binary-string-alternating/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def minFlips(self, s: str) -> int:
        ...
```

Lộ trình:
1. **BFS vét cạn trên mọi chuỗi** đạt được bằng hai thao tác (định nghĩa thuần, chuỗi rất ngắn) — để chứng minh bằng máy rằng chỉ cần "xoay trước, lật sau".
2. **Thử mọi phép xoay** (`n` cách), mỗi cách đếm sai khác với hai mẫu xen kẽ `0101…` và `1010…` (`O(n²)`).
3. **Nhân đôi chuỗi `s + s` + cửa sổ trượt độ dài `n`** — đếm sai khác trong `O(1)` mỗi bước (`O(n)`).
4. **Tổng tiền tố** số vị trí sai khác trên chuỗi nhân đôi (`O(n)`).
5. **Quan sát chẵn/lẻ**: nếu `n` chẵn, xoay không giúp gì ⇒ chỉ cần đếm một lần; nếu `n` lẻ, xoay tương đương "tách chuỗi thành hậu tố + tiền tố" ⇒ tiền tố/hậu tố số sai khác, bộ nhớ `O(1)` thêm.

## 1. Mục tiêu học

1. Nhận ra **thứ tự thao tác** không quan trọng: mọi dãy thao tác tương đương "xoay một lần (bất kỳ số bước) rồi lật" — Type-1 không tốn chi phí.
2. Biểu diễn mọi phép xoay của `s` như **các đoạn độ dài `n` của `s + s`** — mẹo kinh điển cho bài toán chuỗi vòng.
3. Đếm số vị trí khác với mẫu xen kẽ trên **cửa sổ trượt** và hiểu vì sao chỉ cần so với **hai mẫu** cố định độ dài `2n`.
4. Phân tích **chẵn/lẻ của `n`**: chẵn ⇒ xoay vô ích; lẻ ⇒ xoay có thể giúp.
5. Kiểm chứng bằng BFS vét cạn trên không gian chuỗi.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s`. Được thực hiện hai loại thao tác theo thứ tự bất kỳ:
- **Type-1:** bỏ ký tự ở **đầu** chuỗi và nối nó vào **cuối** (xoay trái 1).
- **Type-2:** chọn một ký tự bất kỳ và **lật** giá trị (`'0' ↔ '1'`).

Trả về số thao tác **Type-2 ít nhất** để `s` trở thành **xen kẽ** (không có hai ký tự kề nhau bằng nhau). Ví dụ `"010"`, `"1010"` xen kẽ; `"0100"` không.

**Ví dụ 1:** `s = "111000"` → `2`. Xoay 2 lần thành `"100011"`, lật ký tự thứ 3 và thứ 6 ⇒ `"101010"`.

**Ví dụ 2:** `s = "010"` → `0`.

**Ví dụ 3:** `s = "1110"` → `1`. Lật ký tự thứ 2 ⇒ `"1010"`.

**Ràng buộc:** `1 ≤ s.length ≤ 10⁵`, `s[i] ∈ {'0', '1'}`.

In [1]:
import random
import time
from collections import deque
from itertools import accumulate
from typing import List

examples = [("111000", 2), ("010", 0), ("1110", 1)]
for s, want in examples:
    print(f"s={s!r} -> kỳ vọng {want}")

s='111000' -> kỳ vọng 2
s='010' -> kỳ vọng 0
s='1110' -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Hai chuỗi xen kẽ độ dài `n`

Chỉ có đúng hai chuỗi xen kẽ độ dài `n`: bắt đầu bằng `'0'` (`0101…`) hoặc bằng `'1'` (`1010…`). Ký tự ở vị trí `i` của mẫu thứ nhất là `str(i % 2)`.

In [2]:
def pattern(n, start):
    return "".join(str((i + start) % 2) for i in range(n))
print(pattern(5, 0), pattern(5, 1))

01010 10101


### 3.2 Đếm số vị trí khác mẫu

Số lật tối thiểu để biến `t` thành mẫu `p` = số vị trí `t[i] != p[i]` (khoảng cách Hamming). Vì `p0` và `p1` là bù của nhau: `diff(t, p1) = n − diff(t, p0)`.

In [3]:
def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))
t = "1110"
d0, d1 = hamming(t, pattern(4, 0)), hamming(t, pattern(4, 1))
print(d0, d1, "tổng =", d0 + d1, "= n")

3 1 tổng = 4 = n


### 3.3 Mọi phép xoay là một đoạn của `s + s`

Xoay trái `k` lần cho `s[k:] + s[:k]` = `(s + s)[k : k + n]`.

In [4]:
s = "111000"; ss = s + s
print([ss[k:k + len(s)] for k in range(len(s))])

['111000', '110001', '100011', '000111', '001110', '011100']


### 3.4 Cửa sổ trượt độ dài cố định

Khi cửa sổ dịch từ `[k, k + n − 1]` sang `[k + 1, k + n]`: bỏ đóng góp của vị trí `k`, thêm đóng góp của vị trí `k + n`. So sánh với mẫu xen kẽ **độ dài `2n`** (cố định theo chỉ số tuyệt đối) — chỉ cần chú ý mẫu "bắt đầu ở `k`" có thể là `p0` hoặc `p1` tùy chẵn/lẻ của `k`, nhưng vì ta xét **cả hai** mẫu nên không cần quan tâm.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `s = "111000"` (`n = 6`): thử mọi phép xoay, đếm sai khác với `010101` và `101010`:

| k | xoay | vs 010101 | vs 101010 | min |
|---|---|---|---|---|
| 0 | 111000 | 4 | **2** | 2 |
| 1 | 110001 | 2 | 4 | 2 |
| 2 | 100011 | 4 | 2 | 2 |
| 3 | 000111 | 2 | 4 | 2 |
| 4 | 001110 | 4 | 2 | 2 |
| 5 | 011100 | 2 | 4 | 2 |

Đáp án `2`. Thú vị: **mọi** phép xoay đều cho `2` — vì `n = 6` chẵn (mục 6 giải thích). Ví dụ của đề xoay 2 lần chỉ là một cách; không xoay mà lật vị trí 2 và 5 (`111000 → 101010`) cũng được.

**Vì sao "xoay trước, lật sau" là đủ?** Lật một ký tự rồi xoay ≡ xoay rồi lật ký tự đó ở vị trí mới. Do đó có thể dồn mọi Type-1 lên trước; nhiều Type-1 liên tiếp = một phép xoay `k` bước (`k mod n`).

In [5]:
s = "111000"; n = len(s); ss = s + s
for k in range(n):
    t = ss[k:k + n]
    a, b = hamming(t, pattern(n, 0)), hamming(t, pattern(n, 1))
    print(f"k={k}: {t} | vs 0101..: {a} | vs 1010..: {b} | min {min(a, b)}")

k=0: 111000 | vs 0101..: 4 | vs 1010..: 2 | min 2
k=1: 110001 | vs 0101..: 2 | vs 1010..: 4 | min 2
k=2: 100011 | vs 0101..: 4 | vs 1010..: 2 | min 2
k=3: 000111 | vs 0101..: 2 | vs 1010..: 4 | min 2
k=4: 001110 | vs 0101..: 4 | vs 1010..: 2 | min 2
k=5: 011100 | vs 0101..: 2 | vs 1010..: 4 | min 2


## 5. Cách 1 — Brute Force / baseline

### 5.1 BFS trên mọi trạng thái (định nghĩa thuần)

Trạng thái = chuỗi hiện tại; cạnh Type-1 chi phí 0, Type-2 chi phí 1 ⇒ **0-1 BFS** (deque: cạnh 0 đẩy đầu, cạnh 1 đẩy cuối). Tìm chi phí nhỏ nhất tới một chuỗi xen kẽ bất kỳ. Không giả định gì về thứ tự thao tác.

In [6]:
def is_alternating(t):
    return all(t[i] != t[i + 1] for i in range(len(t) - 1))

def min_flips_bfs(s):
    dist = {s: 0}; dq = deque([s])
    while dq:
        u = dq.popleft(); d = dist[u]
        if is_alternating(u):
            return d
        v = u[1:] + u[0]                             # Type-1, chi phí 0
        if dist.get(v, 10**9) > d:
            dist[v] = d; dq.appendleft(v)
        for i in range(len(u)):                      # Type-2, chi phí 1
            w = u[:i] + ("1" if u[i] == "0" else "0") + u[i + 1:]
            if dist.get(w, 10**9) > d + 1:
                dist[w] = d + 1; dq.append(w)
    return -1

for s, want in examples:
    print(min_flips_bfs(s), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


0-1 BFS lấy ra đỉnh theo khoảng cách không giảm nên lần đầu gặp chuỗi xen kẽ là tối ưu. Số trạng thái tới `2ⁿ` ⇒ chỉ dùng `n ≤ 12`.

### 5.2 Cách 1 (baseline thật) — thử mọi phép xoay, đếm sai khác

`n` phép xoay × `O(n)` đếm ⇒ `O(n²)`.

In [7]:
def min_flips_rotations(s):
    n = len(s)
    p0 = pattern(n, 0)
    best = n
    for k in range(n):
        t = s[k:] + s[:k]
        d0 = sum(1 for x, y in zip(t, p0) if x != y)
        best = min(best, d0, n - d0)
    return best

for s, want in examples:
    print(min_flips_rotations(s), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


**Độ phức tạp:** Time `O(n²)` (`10¹⁰` với `n = 10⁵`), Space `O(n)`.

**Nút thắt:** hai phép xoay liền nhau chỉ khác nhau "ở hai đầu" — nhưng mẫu so sánh lại **dịch pha** (vị trí `i` của `t` mới ứng với vị trí `i + 1` cũ). Giải pháp: so với mẫu cố định theo **chỉ số tuyệt đối** trên `s + s`.

In [8]:
for n in [1000, 3000]:
    s = "".join(random.choice("01") for _ in range(n))
    t0 = time.perf_counter(); min_flips_rotations(s)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=1000: 0.029s


n=3000: 0.251s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (chuẩn hóa thao tác):** Type-1 và Type-2 giao hoán theo nghĩa "lật ký tự `c` rồi xoay" ≡ "xoay rồi lật cùng ký tự `c`" ⇒ đáp án = `min_k min(diff(rot_k(s), P0), diff(rot_k(s), P1))`.

**Insight 2 (mẫu theo chỉ số tuyệt đối):** gọi `T = s + s`, `Q0 = 0101…` độ dài `2n` (`Q0[j] = j % 2`). Cửa sổ `T[k : k + n]` so với `Q0[k : k + n]` — đây là **một trong hai** mẫu xen kẽ độ dài `n` (là `P0` nếu `k` chẵn, `P1` nếu `k` lẻ). Mẫu còn lại cho sai khác `n − (…)`. Nên:

`ans = min_k min(w_k, n − w_k)` với `w_k = #{ j ∈ [k, k + n − 1] : T[j] ≠ Q0[j] }`.

**Cửa sổ trượt:** `w_{k+1} = w_k − mis[k] + mis[k + n]` với `mis[j] = (T[j] ≠ Q0[j])`.

**Insight 3 (chẵn/lẻ):** nếu `n` chẵn, `T[k + n] = T[k]` và `Q0[k + n] = Q0[k]` (vì `n` chẵn) ⇒ `mis[k + n] = mis[k]` ⇒ `w` không đổi khi xoay — **xoay vô ích**, đáp án `min(w_0, n − w_0)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `s + s` + cửa sổ trượt

In [9]:
def min_flips_window(s):
    n = len(s)
    T = s + s
    mis = [1 if T[j] != "01"[j % 2] else 0 for j in range(2 * n)]
    w = sum(mis[:n])
    best = min(w, n - w)
    for k in range(1, n):
        w += mis[k + n - 1] - mis[k - 1]
        best = min(best, w, n - w)
    return best

for s, want in examples:
    print(min_flips_window(s), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** Time `O(n)`, Space `O(n)` (mảng `mis` độ dài `2n`).

### 7.2 Cách 3 — Tổng tiền tố trên `mis`

`P[j] = mis[0] + … + mis[j − 1]` ⇒ `w_k = P[k + n] − P[k]`. Tương đương Cách 2 nhưng thể hiện rõ "đếm trên đoạn".

In [10]:
def min_flips_prefix(s):
    n = len(s)
    T = s + s
    P = [0] + list(accumulate(1 if T[j] != "01"[j % 2] else 0 for j in range(2 * n)))
    best = n
    for k in range(n):
        w = P[k + n] - P[k]
        best = min(best, w, n - w)
    return best

for s, want in examples:
    print(min_flips_prefix(s), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### 7.3 Cách 4 — Không nhân đôi chuỗi: chỉ số `j mod n`

Duyệt `j = 0..2n−1` và đọc `s[j % n]`, giữ cửa sổ bằng biến đếm; bỏ phần tử `j − n` khi `j ≥ n`. Không tạo `s + s` hay mảng `mis` — bộ nhớ phụ `O(1)`.

In [11]:
def min_flips_mod(s):
    n = len(s)
    w = 0
    best = n
    for j in range(2 * n):
        w += s[j % n] != "01"[j % 2]
        if j >= n:
            w -= s[(j - n) % n] != "01"[(j - n) % 2]
        if j >= n - 1:
            best = min(best, w, n - w)
    return best

for s, want in examples:
    print(min_flips_mod(s), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


**Độ phức tạp:** Time `O(n)`, Space `O(1)` phụ.

### 7.4 Cách 5 — Chẵn/lẻ + tiền tố/hậu tố (góc nhìn "cắt chuỗi")

- `n` chẵn ⇒ đáp án `min(w₀, n − w₀)` (Insight 3).
- `n` lẻ: xoay `k` bước cho `s[k:] + s[:k]`. So với mẫu bắt đầu ở `s[k:]` theo **pha gốc** của `s`: phần `s[k:]` giữ nguyên pha (vị trí `j` vẫn đối chiếu `Q0[j]`), phần `s[:k]` được nối ra sau, ở chỉ số tuyệt đối `j + n` của `s + s`; vì `n` lẻ, `Q0[j + n] = 1 − Q0[j]` — tức phần này đối chiếu với mẫu **bù**. Gọi `pre0[k]` = sai khác của `s[:k]` với `Q0`, `suf0[k]` = sai khác của `s[k:]` với `Q0`; khi đó số sai khác của phép xoay với một mẫu là `suf0[k] + (k − pre0[k])`, mẫu kia là `n −` giá trị đó.

In [12]:
def min_flips_parity(s):
    n = len(s)
    mis = [s[j] != "01"[j % 2] for j in range(n)]
    total = sum(mis)
    best = min(total, n - total)
    if n % 2 == 0:
        return best
    pre = 0                                    # pre0[k] = số sai khác của s[:k] với Q0
    for k in range(1, n):
        pre += mis[k - 1]
        suf = total - pre                      # suf0[k] = số sai khác của s[k:] với Q0
        w = suf + (k - pre)                    # phần đầu s[:k] bị lệch pha -> đếm với mẫu bù
        best = min(best, w, n - w)
    return best

for s, want in examples:
    print(min_flips_parity(s), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0
1 kỳ vọng 1


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (có thể `O(1)` nếu tính `mis` tại chỗ hai lượt). Mục 12 kiểm chứng tương đương với các cách khác.

## 8. TRACE từng bước

Trace Cách 2 trên `s = "111000"`: `T = "111000111000"`, `Q0 = "010101010101"`, `mis = [1,0,1,1,0,1, 1,0,1,1,0,1]`:

In [13]:
def trace_window(s):
    n = len(s); T = s + s
    Q0 = "".join("01"[j % 2] for j in range(2 * n))
    mis = [1 if T[j] != Q0[j] else 0 for j in range(2 * n)]
    print("T   =", T); print("Q0  =", Q0); print("mis =", "".join(map(str, mis)))
    w = sum(mis[:n]); best = min(w, n - w)
    print(f"k=0: cửa sổ {T[:n]}, w={w}, n-w={n - w}, best={best}")
    for k in range(1, n):
        out, inn = mis[k - 1], mis[k + n - 1]
        w += inn - out; best = min(best, w, n - w)
        print(f"k={k}: cửa sổ {T[k:k + n]}, ra mis[{k - 1}]={out}, vào mis[{k + n - 1}]={inn}, w={w}, n-w={n - w}, best={best}")
    return best

trace_window("111000")

T   = 111000111000
Q0  = 010101010101
mis = 101101101101
k=0: cửa sổ 111000, w=4, n-w=2, best=2
k=1: cửa sổ 110001, ra mis[0]=1, vào mis[6]=1, w=4, n-w=2, best=2
k=2: cửa sổ 100011, ra mis[1]=0, vào mis[7]=0, w=4, n-w=2, best=2
k=3: cửa sổ 000111, ra mis[2]=1, vào mis[8]=1, w=4, n-w=2, best=2
k=4: cửa sổ 001110, ra mis[3]=1, vào mis[9]=1, w=4, n-w=2, best=2
k=5: cửa sổ 011100, ra mis[4]=0, vào mis[10]=0, w=4, n-w=2, best=2


2

In [14]:
trace_window("10010")      # n lẻ: xoay có tác dụng

T   = 1001010010
Q0  = 0101010101
mis = 1100000111
k=0: cửa sổ 10010, w=2, n-w=3, best=2
k=1: cửa sổ 00101, ra mis[0]=1, vào mis[5]=0, w=1, n-w=4, best=1
k=2: cửa sổ 01010, ra mis[1]=1, vào mis[6]=0, w=0, n-w=5, best=0
k=3: cửa sổ 10100, ra mis[2]=0, vào mis[7]=1, w=1, n-w=4, best=0
k=4: cửa sổ 01001, ra mis[3]=0, vào mis[8]=1, w=2, n-w=3, best=0


0

Với `n` lẻ (`"10010"`), giá trị `w` thay đổi khi trượt — xoay giúp giảm số lật. Với `n` chẵn (`"111000"`), `w` giữ nguyên `4` ở mọi `k` nên đáp án `min(4, 6 − 4) = 2` — xoay không đổi gì. Cell dưới kiểm tra trực tiếp tính tuần hoàn của `mis`:

In [15]:
s = "111000"; n = len(s); T = s + s
mis = [1 if T[j] != "01"[j % 2] else 0 for j in range(2 * n)]
print("mis[k] == mis[k+n] với mọi k?", all(mis[k] == mis[k + n] for k in range(n)))
print("w_k theo k:", [sum(mis[k:k + n]) for k in range(n)])
for t in ["111000", "1110", "10010"]:
    print(t, "| xoay + đếm:", min_flips_rotations(t), "| cửa sổ:", min_flips_window(t), "| BFS:", min_flips_bfs(t),
          "| n chẵn?", len(t) % 2 == 0)

mis[k] == mis[k+n] với mọi k? True
w_k theo k: [4, 4, 4, 4, 4, 4]
111000 | xoay + đếm: 2 | cửa sổ: 2 | BFS: 2 | n chẵn? True
1110 | xoay + đếm: 1 | cửa sổ: 1 | BFS: 1 | n chẵn? True
10010 | xoay + đếm: 0 | cửa sổ: 0 | BFS: 0 | n chẵn? False


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minFlips(self, s: str) -> int:
        n = len(s)
        w = 0
        best = n
        for j in range(2 * n):
            if s[j % n] != "01"[j & 1]:
                w += 1
            if j >= n and s[j - n] != "01"[(j - n) & 1]:
                w -= 1
            if j >= n - 1:
                if w < best:
                    best = w
                if n - w < best:
                    best = n - w
        return best

sol = Solution()
for s, want in examples:
    got = sol.minFlips(s)
    print(got, "OK" if got == want else "SAI")

2 OK
0 OK
1 OK


## 10. Vì sao thuật toán đúng

**(a) Chuẩn hóa.** Một dãy thao tác bất kỳ: mỗi Type-2 lật một **ký tự cụ thể** (đi theo ký tự khi xoay). Hoán đổi một Type-2 với Type-1 đứng ngay sau nó không đổi kết quả (lật ký tự `c` rồi xoay = xoay rồi lật `c` ở vị trí mới). Lặp lại ⇒ mọi Type-1 lên trước. Tổng `k` lần Type-1 = xoay `k mod n`. Sau đó cần ít nhất `diff(rot_k(s), P)` lần lật để thành mẫu xen kẽ `P`, và đúng bằng đó là đủ. ⇒ `ans = min_k min_P diff(rot_k(s), P)`.

**(b) Cửa sổ.** `rot_k(s) = T[k : k+n]`. Mẫu `Q0[k : k+n]` là mẫu xen kẽ độ dài `n` (bắt đầu bằng `k % 2`); mẫu kia là phần bù. Do đó `{diff(rot_k, P0), diff(rot_k, P1)} = {w_k, n − w_k}`. ✓ Cửa sổ trượt cập nhật `w_k` đúng vì cửa sổ mới = cửa sổ cũ bỏ `k − 1`, thêm `k + n − 1`.

**(c) `n` chẵn.** `T` tuần hoàn chu kỳ `n`, `Q0` tuần hoàn chu kỳ 2 (nên chu kỳ `n`) ⇒ `mis` tuần hoàn chu kỳ `n` ⇒ `w_k` hằng số ⇒ xoay không giúp. Điều này được kiểm chứng bằng máy ở mục 12.

**(d) Cách 5 (`n` lẻ).** Ta so `rot_k(s) = s[k:] + s[:k]` với mẫu `Q0[k:k+n]` (theo chỉ số tuyệt đối trên `T`): `s[j]` (`j ≥ k`) đối chiếu `Q0[j]` (pha gốc) — đóng góp `suf0[k]`; `s[j]` (`j < k`) nằm ở chỉ số tuyệt đối `j + n` của `T`, đối chiếu `Q0[j + n] = 1 − Q0[j]` (vì `n` lẻ) — đóng góp `k − pre0[k]`. ✓

## 11. Phân tích Time / Space

Gọi `n = len(s) ≤ 10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. 0-1 BFS trạng thái | `O(2ⁿ · n²)` | `O(2ⁿ n)` | `n ≤ 12` |
| 1b. Mọi phép xoay | `O(n²)` | `O(n)` | baseline |
| 2. `s+s` + cửa sổ | `O(n)` | `O(n)` | rõ ràng |
| 3. Tiền tố | `O(n)` | `O(n)` | |
| 4. Chỉ số `mod n` | `O(n)` | `O(1)` | **nộp bài** |
| 5. Chẵn/lẻ + tiền tố/hậu tố | `O(n)` | `O(n)` | hiểu bản chất |

In [17]:
random.seed(1)
s_big = "".join(random.choice("01") for _ in range(10**5))
s_big_odd = s_big[:-1]
for name, fn in [("window", min_flips_window), ("prefix", min_flips_prefix), ("mod", min_flips_mod),
                 ("parity", min_flips_parity), ("submit", Solution().minFlips)]:
    t0 = time.perf_counter(); a = fn(s_big); b = fn(s_big_odd)
    print(f"{name:7s}: {a} {b}  {time.perf_counter() - t0:.3f}s")

window : 49906 49591  0.035s
prefix : 49906 49591  0.042s
mod    : 49906 49591  0.051s
parity : 49906 49591  0.017s


submit : 49906 49591  0.041s


## 12. Bộ test

So mọi cách với 0-1 BFS (định nghĩa thuần) trên **mọi** chuỗi độ dài `≤ 9`, và với "mọi phép xoay" trên chuỗi dài hơn.

In [18]:
fns = [min_flips_rotations, min_flips_window, min_flips_prefix, min_flips_mod, min_flips_parity,
       lambda s: Solution().minFlips(s)]

for s, want in examples:
    assert min_flips_bfs(s) == want
    for fn in fns:
        assert fn(s) == want

total = 0
for n in range(1, 10):
    for m in range(1 << n):
        s = format(m, f"0{n}b")
        want = min_flips_bfs(s)
        for fn in fns:
            assert fn(s) == want, (fn, s)
        total += 1
print(f"mọi {total} chuỗi độ dài 1..9: khớp 0-1 BFS")

mọi 1022 chuỗi độ dài 1..9: khớp 0-1 BFS


In [19]:
random.seed(2024)
for _ in range(300):
    n = random.randint(10, 300)
    s = "".join(random.choice("01") for _ in range(n))
    want = min_flips_rotations(s)
    for fn in fns[1:]:
        assert fn(s) == want, (fn, s)
# n chẵn: xoay không giúp -> đáp án = min(w0, n - w0)
for _ in range(300):
    n = 2 * random.randint(1, 100); s = "".join(random.choice("01") for _ in range(n))
    w0 = sum(s[j] != "01"[j % 2] for j in range(n))
    assert Solution().minFlips(s) == min(w0, n - w0)
print("chuỗi dài: các cách khớp; n chẵn: đáp án = min(w0, n - w0)")

chuỗi dài: các cách khớp; n chẵn: đáp án = min(w0, n - w0)


Bài kiểm tra cuối xác nhận Insight 3 bằng máy: với ví dụ 1 (`n = 6` chẵn), `w₀ = 4` nên đáp án `min(4, 2) = 2` mà không cần xoay.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua Type-1 (chỉ lật, không xoay)

In [20]:
def wrong_no_rotation(s):
    n = len(s); w = sum(s[j] != "01"[j % 2] for j in range(n))
    return min(w, n - w)

for s in ["10010", "0100", "110"]:
    print(f"{s}: không xoay = {wrong_no_rotation(s)} | đúng = {Solution().minFlips(s)}")

10010: không xoay = 2 | đúng = 0
0100: không xoay = 1 | đúng = 1
110: không xoay = 1 | đúng = 0


Với `n` chẵn thì đúng (Insight 3), nhưng `n` lẻ thì có thể sai. **Sửa:** xét mọi phép xoay.

### Lỗi 2 — Chỉ so với **một** mẫu xen kẽ

In [21]:
def wrong_one_pattern(s):
    n = len(s); T = s + s; best = n
    for k in range(n):
        w = sum(T[k + i] != "01"[i % 2] for i in range(n))      # chỉ mẫu 0101...
        best = min(best, w)
    return best

for s in ["1", "101", "110", "11011", "10", "111000"]:
    print(f"{s}: một mẫu = {wrong_one_pattern(s)} | đúng = {Solution().minFlips(s)}")

1: một mẫu = 1 | đúng = 0
101: một mẫu = 1 | đúng = 0
110: một mẫu = 1 | đúng = 0
11011: một mẫu = 2 | đúng = 1
10: một mẫu = 0 | đúng = 0
111000: một mẫu = 2 | đúng = 2


Thú vị: với `n` **chẵn**, xoay 1 bước biến mẫu `0101…` thành `1010…` nên "một mẫu + mọi phép xoay" vẫn đủ (output cho `"10"`, `"111000"` đúng). Nhưng với `n` **lẻ** thì không: `"101"` đã xen kẽ (0 lật) mà chỉ so với mẫu `010` cho kết quả 1. **Sửa:** luôn xét cả `w` và `n − w`.

### Lỗi 3 — Cửa sổ trượt dùng mẫu **tương đối** (đặt lại pha theo đầu cửa sổ) nhưng cập nhật như mẫu tuyệt đối

In [22]:
def wrong_relative_phase(s):
    n = len(s); T = s + s
    w = sum(T[i] != "01"[i % 2] for i in range(n)); best = min(w, n - w)
    for k in range(1, n):
        w += (T[k + n - 1] != "01"[(n - 1) % 2]) - (T[k - 1] != "01"[0])   # SAI: pha tương đối, w không còn đúng
        best = min(best, w, n - w)
    return best

random.seed(3)
for _ in range(5000):
    s = "".join(random.choice("01") for _ in range(random.randint(2, 9)))
    if wrong_relative_phase(s) != Solution().minFlips(s):
        print(f"phản ví dụ: {s!r} | sai = {wrong_relative_phase(s)} | đúng = {Solution().minFlips(s)}")
        break

phản ví dụ: '01100' | sai = 2 | đúng = 1


Khi cửa sổ dịch, mẫu tương đối cũng **dịch pha** ⇒ mọi vị trí bên trong đổi đóng góp, không chỉ hai đầu. **Sửa:** so với mẫu theo **chỉ số tuyệt đối** `j % 2` trên `T`.

### Lỗi 4 — Nghĩ rằng cần đếm cả Type-1

In [23]:
def wrong_count_rotations(s):
    n = len(s); T = s + s; best = 10**9
    for k in range(n):
        w = sum(T[k + i] != "01"[(k + i) % 2] for i in range(n))
        best = min(best, k + min(w, n - w))      # SAI: Type-1 không tính vào đáp án
    return best

for t in ["111000", "10010", "11011"]:
    print(f"{t}: sai = {wrong_count_rotations(t)} | đúng = {Solution().minFlips(t)}")

111000: sai = 2 | đúng = 2
10010: sai = 2 | đúng = 0
11011: sai = 2 | đúng = 1


Đề chỉ hỏi số thao tác **Type-2**. **Sửa:** không cộng `k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chuỗi/mảng **vòng** (được xoay tự do) ⇒ xét `s + s` và mọi đoạn độ dài `n`.
- "Biến thành chuỗi xen kẽ" ⇒ chỉ có hai mẫu, và chúng **bù nhau**: `diff₁ = n − diff₀`.
- Cửa sổ trượt với mẫu **cố định theo chỉ số tuyệt đối** để cập nhật `O(1)`.
- Thao tác miễn phí giao hoán với thao tác tốn phí ⇒ dồn thao tác miễn phí lên trước.
- Kiểm tra chẵn/lẻ của độ dài — thường làm đơn giản bài toán vòng.
- Họ bài: 1758 *Minimum Changes To Make Alternating Binary String* (không xoay), 2134 *Minimum Swaps to Group All 1's Together II* (mảng vòng + cửa sổ), 918 *Maximum Sum Circular Subarray*.

## 15. Biến thể của bài

1. **1758 (không có Type-1):** `min(w₀, n − w₀)`.
2. **Trả về phép xoay tối ưu `k`** và chuỗi kết quả.
3. **Chi phí lật khác nhau theo vị trí** (`cost[i]` gắn với ký tự): cửa sổ trượt trên tổng chi phí sai khác.
4. **Xoay có chi phí `c` mỗi bước**, tối thiểu tổng chi phí: `min_k (c · k' + flips(k))` với `k' = min(k, n − k)` nếu được xoay cả hai chiều.
5. **Mẫu đích là chuỗi tuần hoàn khác** (ví dụ `001001…`): cùng kỹ thuật, nhiều mẫu hơn.

In [24]:
def best_rotation(s):
    """Biến thể 2: (số lật, k, chuỗi đích)."""
    n = len(s); T = s + s; best = None
    for k in range(n):
        t = T[k:k + n]
        for start in (0, 1):
            p = pattern(n, start); d = hamming(t, p)
            if best is None or d < best[0]:
                best = (d, k, p)
    return best

def min_flips_weighted(s, cost):
    """Biến thể 3: cost[i] là chi phí lật ký tự gốc s[i]; trả về chi phí nhỏ nhất."""
    n = len(s); best = None
    for k in range(n):
        idx = [(k + i) % n for i in range(n)]
        for start in (0, 1):
            c = sum(cost[j] for i, j in enumerate(idx) if s[j] != str((i + start) % 2))
            best = c if best is None else min(best, c)
    return best

for s, _ in examples:
    print(s, "->", best_rotation(s))
print("có trọng số:", min_flips_weighted("111000", [1, 5, 1, 1, 5, 1]))

111000 -> (2, 0, '101010')
010 -> (0, 0, '010')
1110 -> (1, 0, '1010')
có trọng số: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. 0-1 BFS | định nghĩa thuần | mũ | mũ | kiểm chứng |
| 1b. Mọi phép xoay | đếm lại từ đầu | `O(n²)` | `O(n)` | baseline |
| 2. `s+s` + cửa sổ | mẫu tuyệt đối | `O(n)` | `O(n)` | rõ ràng |
| 3. Tiền tố | `P[k+n] − P[k]` | `O(n)` | `O(n)` | |
| 4. `mod n` | không nhân đôi | `O(n)` | `O(1)` | **nộp bài** |
| 5. Chẵn/lẻ | cắt tiền tố/hậu tố | `O(n)` | `O(n)` | hiểu bản chất |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `minFlips("01001001101")` và `minFlips("1")`?

**Bài 2 (tính tay):** với `s = "111000"`, tính `mis` và `w_k` cho mọi `k`, rồi giải thích **bằng số liệu** vì sao đáp án là `2` và vì sao xoay không thay đổi kết quả.

**Bài 3 (điền TODO):** hoàn thiện `flips_without_rotation(s)` (bài 1758).

**Bài 4 (tự viết lại):** viết lại Cách 4 trả về thêm `k` tối ưu (số lần Type-1 cần dùng, `0 ≤ k < n`).

**Bài 5 (mini challenge):** biến thể 4 — mỗi lần Type-1 tốn `c` (chỉ xoay trái), mỗi lần lật tốn 1. Tìm chi phí nhỏ nhất. Kiểm chứng bằng 0-1 BFS tổng quát (Dijkstra) trên chuỗi ngắn. (Biến thể tự đặt.)

In [25]:
def flips_without_rotation(s):
    # TODO: w0 = số vị trí khác 0101...; trả về min(w0, n - w0)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `n = 11` lẻ ⇒ xoay có thể giúp. (2) Chạy `trace_window`. (3) Đáp án theo code: hãy đối chiếu với `min_flips_rotations`. `"1"` xen kẽ sẵn ⇒ `0`.

**Bài 2:** (1) `T = "111000111000"`, `Q0 = "010101010101"`. (2) `mis` = vị trí `T[j] ≠ Q0[j]`; đếm trong `[0, 5]`. (3) `w_k = 4` với mọi `k` ⇒ đáp án `min(4, 2) = 2`.

**Bài 3:** (1) `w0 = sum(s[j] != "01"[j % 2] …)`. (2) `return min(w0, n − w0)`. (3) Thử `"0100"` ⇒ `1`.

**Bài 4:** (1) Khi `best` giảm, ghi `k = j − n + 1`. (2) Cẩn thận cửa sổ đầu tiên (`j = n − 1`) ứng với `k = 0`. (3) Kiểm tra với `best_rotation`.

**Bài 5:** (1) Chi phí `c · k + flips(k)` với `k ∈ [0, n)`. (2) Dùng mảng `w_k` từ Cách 2. (3) Dijkstra trên trạng thái chuỗi cho `n ≤ 8`.

## 19. Đáp án tham khảo

In [26]:
import heapq

# Bài 1
print("Bài 1:", Solution().minFlips("01001001101"), Solution().minFlips("1"))
assert Solution().minFlips("01001001101") == min_flips_rotations("01001001101") and Solution().minFlips("1") == 0

# Bài 2
s = "111000"; n = 6; T = s + s
mis = [int(T[j] != "01"[j % 2]) for j in range(2 * n)]
ws = [sum(mis[k:k + n]) for k in range(n)]
print("Bài 2: mis =", mis, "| w_k =", ws, "| đáp án =", min(ws[0], n - ws[0]))
assert len(set(ws)) == 1 and min(ws[0], n - ws[0]) == 2

# Bài 3
def flips_without_rotation_ans(s):
    n = len(s); w0 = sum(s[j] != "01"[j % 2] for j in range(n))
    return min(w0, n - w0)
assert flips_without_rotation_ans("0100") == 1 and flips_without_rotation_ans("10") == 0

# Bài 4
def min_flips_with_k(s):
    n = len(s); w = 0; best = (n + 1, 0)
    for j in range(2 * n):
        w += s[j % n] != "01"[j % 2]
        if j >= n:
            w -= s[j - n] != "01"[(j - n) % 2]
        if j >= n - 1:
            best = min(best, (min(w, n - w), j - n + 1))
    return best

random.seed(201)
for _ in range(500):
    s = "".join(random.choice("01") for _ in range(random.randint(1, 15)))
    f, k = min_flips_with_k(s)
    t = s[k:] + s[:k]
    assert f == Solution().minFlips(s) == min(hamming(t, pattern(len(s), 0)), hamming(t, pattern(len(s), 1)))
print("Bài 1-4: OK")

Bài 1: 2 0
Bài 2: mis = [1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1] | w_k = [4, 4, 4, 4, 4, 4] | đáp án = 2
Bài 1-4: OK


In [27]:
# Bài 5 — xoay tốn c mỗi bước
def min_cost_rotate(s, c):
    n = len(s); T = s + s; best = None
    mis = [int(T[j] != "01"[j % 2]) for j in range(2 * n)]
    w = sum(mis[:n])
    for k in range(n):
        if k > 0:
            w += mis[k + n - 1] - mis[k - 1]
        cost = c * k + min(w, n - w)
        best = cost if best is None else min(best, cost)
    return best

def min_cost_dijkstra(s, c):
    dist = {s: 0}; pq = [(0, s)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        if is_alternating(u): return d
        nbrs = [(u[1:] + u[0], c)] + [(u[:i] + ("1" if u[i] == "0" else "0") + u[i + 1:], 1) for i in range(len(u))]
        for v, w_ in nbrs:
            if d + w_ < dist.get(v, 10**9):
                dist[v] = d + w_; heapq.heappush(pq, (d + w_, v))

random.seed(202)
for _ in range(300):
    s = "".join(random.choice("01") for _ in range(random.randint(1, 8))); c = random.randint(0, 3)
    assert min_cost_rotate(s, c) == min_cost_dijkstra(s, c), (s, c)
print("Bài 5: khớp Dijkstra trên trạng thái chuỗi")

Bài 5: khớp Dijkstra trên trạng thái chuỗi


## 20. Template nhớ nhanh + Checklist

```python
# Chuỗi vòng + mẫu xen kẽ: cửa sổ độ dài n trên s+s, so với "01"[j % 2] theo chỉ số tuyệt đối
w, best = 0, n
for j in range(2 * n):
    w += s[j % n] != "01"[j % 2]                          # vào
    if j >= n: w -= s[j - n] != "01"[(j - n) % 2]         # ra
    if j >= n - 1: best = min(best, w, n - w)             # hai mẫu bù nhau
return best
```

**Checklist trước khi submit:**
- [ ] Xét **cả hai** mẫu (`w` và `n − w`).
- [ ] Mẫu so sánh theo **chỉ số tuyệt đối** trên chuỗi nhân đôi.
- [ ] Không cộng số lần Type-1 vào đáp án.
- [ ] `n` chẵn: xoay không đổi kết quả (dùng để tự kiểm tra).
- [ ] Thử `n = 1` (đáp án 0).